# Ekstraksi Fitur (Feature Engineering) untuk ML Klasik

Sesuai arahan di `notes.txt`, kita akan mengekstrak dua jenis fitur berbeda untuk dievaluasi secara terpisah pada Machine Learning Klasik:
1. **Canny Edge Detection** (Deteksi Tepi)
2. **Discrete Wavelet Transform / DWT** (Transformasi Wavelet)

Langkah-langkah yang dilakukan:
- Load citra yang sudah dipraproses dari `data/results/`
- Ekstraksi fitur (Canny dan DWT)
- Flattening menjadi array 1D
- Feature Scaling / Normalization (StandardScaler)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
import pywt  # pip install PyWavelets

plt.rcParams['figure.figsize'] = (10, 4)

## 1. Definisi Fungsi Ekstraksi Fitur

Kedua fitur ini tidak digabung (concatenated). Kita akan menyiapkan dua dataset terpisah: `X_canny` dan `X_dwt`.

In [ ]:
def extract_canny(img):
    """
    Menggunakan Canny Edge Detection.
    Menerima gambar Grayscale numpy array.
    """
    edges = cv2.Canny(img, threshold1=100, threshold2=200)
    # Flatten jadi 1D array
    return edges.flatten()

def extract_dwt(img, wavelet='haar'):
    """
    Menggunakan Discrete Wavelet Transform (DWT).
    Kita ambil LL (Approximation Coefficients) yang menangkap bentuk umum.
    """
    coeffs2 = pywt.dwt2(img, wavelet)
    LL, (LH, HL, HH) = coeffs2
    # Flatten LL jadi 1D array
    return LL.flatten()


## 2. Visualisasi Fitur
Mari kita lihat perbandingan gambar asli (setelah preprocessing) dengan hasil ekstraksi Canny dan DWT.

In [ ]:
# Gunakan salah satu gambar preprocessed dari dataset jika ada
sample_dir = Path("../data/results/processed/Validation/WithMask")
sample_images = list(sample_dir.glob("*.png"))

if len(sample_images) > 0:
    sample_img = cv2.imread(str(sample_images[0]), cv2.IMREAD_GRAYSCALE)
    
    # Visualisasi
    edges_img = cv2.Canny(sample_img, 100, 200)
    coeffs2 = pywt.dwt2(sample_img, 'haar')
    LL, _ = coeffs2
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(sample_img, cmap='gray')
    axes[0].set_title("Original Preprocessed (224x224)")
    
    axes[1].imshow(edges_img, cmap='gray')
    axes[1].set_title("Canny Edge Detection")
    
    axes[2].imshow(LL, cmap='gray')
    axes[2].set_title("DWT (LL Coefficients)")
    
    for ax in axes: ax.axis('off')
    plt.show()
else:
    print("Jalankan notebook 02_preprocesses.ipynb terlebih dahulu untuk men-generate data sample.")

## 3. Ekstraksi Fitur Secara Massal
Looping melalui struktur folder `data/results` untuk membangun dataset (X dan y).

In [ ]:
def load_and_extract_features(root_dir, require_preprocess=False):
    X_canny_list, X_dwt_list, y_list = [], [], []
    classes = ["WithMask", "WithoutMask", "MaskWornIncorrect"]
    if not Path(root_dir).exists(): return None, None, None

    for cls_idx, cls_name in enumerate(classes):
        cls_dir = Path(root_dir) / cls_name
        if not cls_dir.exists(): continue
            
        images = list(cls_dir.glob("*.png")) + list(cls_dir.glob("*.jpg"))
        for img_path in tqdm(images, desc=f"Loading {cls_name} di {Path(root_dir).name}"):
            img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
            if img is not None:
                if require_preprocess:
                    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
                    img = clahe.apply(img)
                    img = cv2.GaussianBlur(img, (5,5), 0)
                
                img_resized = cv2.resize(img, (128, 128)) # Dikecilkan untuk SVM
                X_canny_list.append(extract_canny(img_resized))
                X_dwt_list.append(extract_dwt(img_resized))
                y_list.append(cls_idx)
                
    return np.array(X_canny_list), np.array(X_dwt_list), np.array(y_list)

RESULTS_ROOT = Path("../data/results")
import kagglehub
KAGGLE_DIR = Path(kagglehub.dataset_download("ashishjangra27/face-mask-12k-images-dataset")) / "Face Mask Dataset"
if not KAGGLE_DIR.exists(): KAGGLE_DIR = KAGGLE_DIR.parent

"""
# 1. Load Train (Non-Augmented) dari Raw Kaggle + Praproses on-the-fly
X_train_unaug_canny, X_train_unaug_dwt, y_train_unaug = load_and_extract_features(KAGGLE_DIR / "Train", require_preprocess=True)

# 2. Load Train (Augmented) dari Lokal
train_aug_dir = RESULTS_ROOT / "augmented" / "Train"
X_train_aug_canny, X_train_aug_dwt, y_train_aug = load_and_extract_features(train_aug_dir)

# 3. Load Validation (Processed)
val_dir = RESULTS_ROOT / "processed" / "Validation"
X_val_canny, X_val_dwt, y_val = load_and_extract_features(val_dir)

# 4. Load Test (Processed)
test_dir = RESULTS_ROOT / "processed" / "Test"
X_test_canny, X_test_dwt, y_test = load_and_extract_features(test_dir)
"""

## 4. Feature Scaling / Normalization
Untuk ML Klasik seperti SVM dan KNN, nilai fitur harus diseragamkan rentangnya (Transformasi Linear). Kita gunakan `StandardScaler`.

In [ ]:
def scale_features(X_train, X_val, X_test):
    if X_train is None: return None, None, None, None
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val) if X_val is not None else None
    X_test_scaled = scaler.transform(X_test) if X_test is not None else None
    return X_train_scaled, X_val_scaled, X_test_scaled, scaler

"""
# Scaling untuk Unaugmented
X_tr_unaug_canny_s, X_v_c, X_te_c, sc_c_unaug = scale_features(X_train_unaug_canny, X_val_canny, X_test_canny)
X_tr_unaug_dwt_s, X_v_d, X_te_d, sc_d_unaug = scale_features(X_train_unaug_dwt, X_val_dwt, X_test_dwt)

# Scaling untuk Augmented
X_tr_aug_canny_s, _, _, sc_c_aug = scale_features(X_train_aug_canny, X_val_canny, X_test_canny)
X_tr_aug_dwt_s, _, _, sc_d_aug = scale_features(X_train_aug_dwt, X_val_dwt, X_test_dwt)
"""

## 5. Simpan Fitur Numpy
Karena ekstraksi DWT atau loading puluhan ribu file bisa memakan waktu, ada baiknya menyimpan array hasil ekstraksi ke format `.npy` atau `.npz`.

In [ ]:
import os

def save_numpy_features(folder_path, prefix, X_train, X_val, X_test, y_train, y_val, y_test):
    os.makedirs(folder_path, exist_ok=True)
    np.savez_compressed(f"{folder_path}/{prefix}_features.npz", 
                        X_train=X_train, X_val=X_val, X_test=X_test,
                        y_train=y_train, y_val=y_val, y_test=y_test)
    print(f"Fitur {prefix} berhasil disimpan ke {folder_path}/{prefix}_features.npz")

"""
# Simpan kedua fitur agar siap diload di 04_modelling.ipynb
save_numpy_features("../data/features", "canny", X_train_canny_s, X_val_canny_s, X_test_canny_s, y_train, y_val, y_test)
save_numpy_features("../data/features", "dwt", X_train_dwt_s, X_val_dwt_s, X_test_dwt_s, y_train, y_val, y_test)
"""